# CrossNet-V_ATN_Gaussian_LR1e-3_WD3e-4

ATN+Gaussian V with conservative LR (1e-3) and lighter weight decay (3e-4); probes regularisation strength.


## Dependencies


In [ ]:
#------------
# Dependencies
#------------

import csv
import json
import math
import random
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm


## Random seed and device


In [ ]:
#-------------
# Random Seed
#-------------

seed = 2025
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


In [ ]:
#------------------
# Computing Device
#------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)


## Utilities


In [ ]:
#---------------------
# Important Functions
#---------------------

def parameter_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def nmse_db(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    with torch.no_grad():
        mse = torch.sum((y_true - y_pred) ** 2, dim=(1, 2, 3))
        power = torch.sum(y_true ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
        return float((10.0 * torch.log10((mse / power).mean().clamp_min(1e-12))).item())


def nmse_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    mse = torch.sum((y_true - y_pred) ** 2, dim=(1, 2, 3))
    power = torch.sum(y_true ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
    return (mse / power).mean()


## Variant configuration


In [ ]:
#----------------------
# Variant Configuration
#----------------------

variant_name = "CrossNet-V_ATN_Gaussian_LR1e-3_WD3e-4"
batch_size = 128
learning_rate = 0.001
weight_decay = 0.0003
eta_min = 1e-05
num_epochs = 1000
scheduler_type = "cosine"
use_sigmoid = False
train_path = "train_data.mat"
val_path = "val_data.mat"
test_path = "test_data.mat"
print("Variant:", variant_name)


## Dataset and loaders


In [ ]:
#-----------------
# Dataset Functions
#-----------------

def _structured_to_tensor(arr):
    real = np.squeeze(arr["real"], axis=2).astype(np.float32)
    imag = np.squeeze(arr["imag"], axis=2).astype(np.float32)
    return torch.from_numpy(np.stack([real, imag], axis=0))


class QuadrigaCSIDataset(Dataset):
    def __init__(self, path, key="csi_dl"):
        self.path = str(path)
        self.key = key
        with h5py.File(self.path, "r") as f:
            self.length = int(f[self.key].shape[3])
        self._file = None
        self._dataset = None

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self._file is None:
            self._file = h5py.File(self.path, "r")
            self._dataset = self._file[self.key]
        return _structured_to_tensor(self._dataset[:, :, :, idx])

    def close(self):
        if self._file is not None:
            self._file.close()
            self._file = None
            self._dataset = None


def iter_h5_batches(path, key="csi_dl", batch_size=64, max_samples=None):
    with h5py.File(path, "r") as f:
        ds = f[key]
        total = int(ds.shape[3])
        if max_samples is not None:
            total = min(total, max_samples)
        for start in range(0, total, batch_size):
            stop = min(start + batch_size, total)
            chunk = ds[:, :, :, start:stop]
            real = np.squeeze(chunk["real"], axis=2).astype(np.float32)
            imag = np.squeeze(chunk["imag"], axis=2).astype(np.float32)
            stacked = np.transpose(np.stack([real, imag], axis=0), (3, 0, 1, 2))
            yield torch.from_numpy(stacked)


In [ ]:
#------------
# DataLoader
#------------

train_data = QuadrigaCSIDataset(train_path)
val_data = QuadrigaCSIDataset(val_path)
test_data = QuadrigaCSIDataset(test_path)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=pin)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=pin)
print(f"Train/Val/Test sizes: {len(train_data)}, {len(val_data)}, {len(test_data)}")


## Normalizer


In [ ]:
#----------------------
# Normalization Helpers
#----------------------

class ChannelMinMaxNormalizer(nn.Module):
    def __init__(self, channel_min, channel_max):
        super().__init__()
        cmin = torch.tensor(list(channel_min), dtype=torch.float32).view(1, 2, 1, 1)
        cmax = torch.tensor(list(channel_max), dtype=torch.float32).view(1, 2, 1, 1)
        self.register_buffer("channel_min", cmin)
        self.register_buffer("channel_range", (cmax - cmin).clamp_min(1e-8))

    def normalize(self, x):
        return (x - self.channel_min) / self.channel_range

    def denormalize(self, x):
        return x * self.channel_range + self.channel_min


class GlobalStdNormalizer(nn.Module):
    def __init__(self, std_value):
        super().__init__()
        v = torch.tensor(float(std_value), dtype=torch.float32).view(1, 1, 1, 1)
        self.register_buffer("std_value", v)

    def normalize(self, x):
        return x / (self.std_value + 1e-8)

    def denormalize(self, x):
        return x * (self.std_value + 1e-8)


def fit_gaussian_normalizer(train_path, batch_size=64):
    sum_sq = 0.0
    count = 0
    for batch in iter_h5_batches(train_path, batch_size=batch_size):
        sum_sq += float((batch ** 2).sum().item())
        count += int(batch.numel())
    return GlobalStdNormalizer(math.sqrt(sum_sq / max(count, 1) + 1e-12))


def fit_minmax_normalizer(train_path, batch_size=64):
    cmin = torch.full((2,), float("inf"))
    cmax = torch.full((2,), float("-inf"))
    for batch in iter_h5_batches(train_path, batch_size=batch_size):
        cmin = torch.minimum(cmin, batch.amin(dim=(0, 2, 3)))
        cmax = torch.maximum(cmax, batch.amax(dim=(0, 2, 3)))
    return ChannelMinMaxNormalizer(cmin.tolist(), cmax.tolist())


## ATN / STN convolutional front-ends


In [ ]:
#-----------------
# ATN / STN Modules
#-----------------

class ATN(nn.Module):
    def __init__(self, use_sigmoid):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, 3, stride=(2, 1), padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.conv2 = nn.Conv2d(16, 16, 3, stride=(2, 1), padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.conv3 = nn.Conv2d(16, 2, 3, stride=(2, 1), padding=1)
        self.bn3 = nn.BatchNorm2d(2)
        self.act = nn.Sigmoid() if use_sigmoid else nn.Identity()

    def forward(self, x):
        # Encode the normalized CSI into a compact latent grid.
        x = self.prelu1(self.bn1(self.conv1(x)))
        x = self.prelu2(self.bn2(self.conv2(x)))
        return self.act(self.bn3(self.conv3(x)))


class STN(nn.Module):
    def __init__(self, use_sigmoid):
        super().__init__()
        self.deconv1 = nn.ConvTranspose2d(2, 16, 3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU(16)
        self.deconv2 = nn.ConvTranspose2d(16, 16, 3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU(16)
        self.deconv3 = nn.ConvTranspose2d(16, 2, 3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn3 = nn.BatchNorm2d(2)
        self.act = nn.Sigmoid() if use_sigmoid else nn.Identity()

    def forward(self, x):
        # Decode the latent grid back to the original resolution.
        x = self.prelu1(self.bn1(self.deconv1(x)))
        x = self.prelu2(self.bn2(self.deconv2(x)))
        return self.act(self.bn3(self.deconv3(x)))


## Transformer building blocks


In [ ]:
#---------------
# Model Modules
#---------------

class MLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, 4 * dim)
        self.fc2 = nn.Linear(4 * dim, dim)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class QKV(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.w_q = nn.Linear(dim, dim // 2, bias=True)
        self.w_k = nn.Linear(dim, dim // 2, bias=True)
        self.w_v = nn.Linear(dim, dim // 2, bias=True)

    def forward(self, x):
        b, n, c = x.shape
        d = c // (2 * self.heads)
        q = self.w_q(x).reshape(b, n, self.heads, d).permute(0, 2, 1, 3).contiguous()
        k = self.w_k(x).reshape(b, n, self.heads, d).permute(0, 2, 1, 3).contiguous()
        v = self.w_v(x).reshape(b, n, self.heads, d).permute(0, 2, 1, 3).contiguous()
        return q, k, v


class MHA(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.scale = (dim // (2 * heads)) ** -0.5
        self.proj = nn.Linear(dim // 2, dim)

    def forward(self, q, k, v):
        b, h, n, d = q.shape
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(b, n, h * d)
        return self.proj(out)


class EncoderBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.qkv = QKV(dim, heads)
        self.mha = MHA(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim)

    def forward(self, x):
        q, k, v = self.qkv(self.norm1(x))
        x = x + self.mha(q, k, v)
        x = x + self.mlp(self.norm2(x))
        return x


class CrossBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.qkv1 = QKV(dim, heads)
        self.mha1 = MHA(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = MLP(dim)
        self.norm3 = nn.LayerNorm(dim)
        self.qkv2 = QKV(dim, heads)
        self.mha2 = MHA(dim, heads)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = MLP(dim)

    def forward(self, x, y):
        qx, kx, vx = self.qkv1(self.norm1(x))
        qy, ky, vy = self.qkv2(self.norm3(y))
        x = x + self.mha1(qx, ky, vy)
        y = y + self.mha2(qy, kx, vx)
        x = x + self.mlp1(self.norm2(x))
        y = y + self.mlp2(self.norm4(y))
        return x, y


## CrossNet-V backbone


In [ ]:
#-------
# Model
#-------

class MiniCrossNetV(nn.Module):
    def __init__(self, codeword=512, heads=2, use_sigmoid=False):
        super().__init__()
        self.seq, self.dim, self.d_model = 64, 32, 16
        self.encsubemb = nn.Linear(self.dim, self.d_model)
        self.encantemb = nn.Linear(self.dim, self.d_model)
        self.module1 = EncoderBlock(self.d_model, heads)
        self.module2 = EncoderBlock(self.d_model, heads)
        self.module3 = CrossBlock(self.d_model, heads)
        self.fusenorm1 = nn.LayerNorm(self.d_model)
        self.reduction = nn.Linear(self.seq * self.d_model, codeword)

        self.expansion = nn.Linear(codeword, self.seq * self.dim)
        self.decsubemb = nn.Linear(self.dim, self.d_model)
        self.decantemb = nn.Linear(self.dim, self.d_model)
        self.module4 = EncoderBlock(self.d_model, heads)
        self.module5 = EncoderBlock(self.d_model, heads)
        self.module6 = CrossBlock(self.d_model, heads)
        self.fusenorm2 = nn.LayerNorm(self.d_model)
        self.outhead = nn.Linear(self.d_model, self.dim)
        self.act = nn.Sigmoid() if use_sigmoid else nn.Identity()

    def forward(self, x):
        b, c, h, w = x.shape
        # Build subcarrier and antenna token sequences.
        sub = torch.cat([x[:, 0, :, :], x[:, 1, :, :]], dim=1)
        ant = torch.cat([x.transpose(-2, -1)[:, 0, :, :], x.transpose(-2, -1)[:, 1, :, :]], dim=1)
        # Encode the normalized CSI into a compact latent grid.
        sub = self.module1(self.encsubemb(sub))
        ant = self.module2(self.encantemb(ant))
        sub, ant = self.module3(sub, ant)
        z = self.reduction(self.fusenorm1(sub + ant).reshape(b, -1))
        # Decode the latent grid back to the original resolution.
        d = self.expansion(z).reshape(b, c, h, w)
        sub = torch.cat([d[:, 0, :, :], d[:, 1, :, :]], dim=1)
        ant = torch.cat([d.transpose(-2, -1)[:, 0, :, :], d.transpose(-2, -1)[:, 1, :, :]], dim=1)
        sub = self.module4(self.decsubemb(sub))
        ant = self.module5(self.decantemb(ant))
        sub, ant = self.module6(sub, ant)
        out = self.act(self.outhead(self.fusenorm2(sub + ant)))
        return torch.stack((out[:, :h, :], out[:, h:, :]), dim=1)


## ATN-CrossNet-STN autoencoder wrapper


In [ ]:
#--------------------
# Autoencoder Wrapper
#--------------------

class ATNCrossNetAutoencoder(nn.Module):
    def __init__(self, normalizer, crossnet, use_sigmoid):
        super().__init__()
        self.normalizer = normalizer
        self.atn = ATN(use_sigmoid)
        self.crossnet = crossnet
        self.stn = STN(use_sigmoid)

    def forward(self, x_raw):
        # Normalize, project, reconstruct, and restore the CSI sample.
        x = self.normalizer.normalize(x_raw)
        x = self.atn(x)
        x = self.crossnet(x)
        x = self.stn(x)
        return self.normalizer.denormalize(x)


## Build model


In [ ]:
#-----------
# Model Setup
#-----------

crossnet = MiniCrossNetV(use_sigmoid=False)
normalizer = fit_gaussian_normalizer(train_path)
model = ATNCrossNetAutoencoder(normalizer, crossnet, use_sigmoid=False).to(device)
# Report the trainable parameter count for the configured variant.
print(f"Trainable parameters: {parameter_count(model):,}")


## Sanity-check forward pass


In [ ]:
#----------------
# Model Smoke Test
#----------------

# Quick shape check before the training loop starts.
sample = next(iter(train_loader)).to(device)
with torch.no_grad():
    out = model(sample)
print("Input :", tuple(sample.shape))
print("Output:", tuple(out.shape))


## Loss, optimiser, scheduler


In [ ]:
#-------------------------------
# Loss, Optimizer, and Scheduler
#-------------------------------

criterion = nn.MSELoss().to(device)
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay, betas=(0.8, 0.98))
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=eta_min)

aux_loss_fn = None


## Training loop


In [ ]:
#-----------------------------
# Model Training and Validation
#-----------------------------

def evaluate_nmse(model, loader):
    # Validation pass
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            total += nmse_db(batch, model(batch))
            n += 1
    return total / max(n, 1)


# Prepare experiment outputs and tracking buffers.
output_dir = Path("outputs") / variant_name
output_dir.mkdir(parents=True, exist_ok=True)

train_losses, val_losses, nmse_scores, lrs = [], [], [], []
best_nmse = float("inf")

for epoch in tqdm(range(num_epochs), desc=f"Training {variant_name}"):
    # Training pass
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(batch)
        loss = criterion(pred, batch)
        if aux_loss_fn is not None:
            loss = loss + aux_loss_fn(model, batch, pred)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            val_loss += criterion(model(batch), batch).item() * batch.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    avg_nmse = evaluate_nmse(model, test_loader)

    # Update the scheduler and record metrics.
    if isinstance(scheduler, ReduceLROnPlateau):
        scheduler.step(val_loss)
    else:
        scheduler.step()
    cur_lr = optimizer.param_groups[0]["lr"]

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    nmse_scores.append(avg_nmse)
    lrs.append(cur_lr)

    print(f"Epoch {epoch + 1}/{num_epochs} | Train: {train_loss:.6e} | Val: {val_loss:.6e} | NMSE: {avg_nmse:.4f} dB | LR: {cur_lr:.2e}")

    # Save the best checkpoint and metric traces.
    if avg_nmse <= best_nmse:
        best_nmse = avg_nmse
        torch.save({"epoch": epoch + 1, "model": model.state_dict()}, output_dir / "best_model.pth")

    np.savetxt(output_dir / "train_losses.csv", np.array(train_losses), delimiter=",")
    np.savetxt(output_dir / "val_losses.csv", np.array(val_losses), delimiter=",")
    np.savetxt(output_dir / "nmse_scores.csv", np.array(nmse_scores), delimiter=",")
    np.savetxt(output_dir / "learning_rates.csv", np.array(lrs), delimiter=",")


## Cleanup


In [ ]:
#----------------
# Dataset Cleanup
#----------------

# Release HDF5 handles opened by the datasets.
for loader in (train_loader, val_loader, test_loader):
    if hasattr(loader.dataset, "close"):
        loader.dataset.close()
print(f"Best test NMSE for {variant_name}: {min(nmse_scores):.4f} dB")
